In [1]:
# Install required libraries (only need to run once)
!pip install -q transformers datasets evaluate rouge_score sentencepiece accelerate

# Now import everything
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import Dataset
import evaluate
import nltk
from tqdm import tqdm

# Download sentence tokenizer for evaluation
nltk.download('punkt', quiet=True)

# Check if GPU is enabled – you should see "True"
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
GPU available: True
GPU name: Tesla T4


In [4]:
import os
print("Files in the dataset:")
for f in os.listdir('/kaggle/input/datasets/sbhatti/news-summarization'):
    print(f" - {f}")

Files in the dataset:
 - data.csv


In [5]:
df = pd.read_csv('/kaggle/input/datasets/sbhatti/news-summarization/data.csv')
df.head(3)

,Unnamed: 0,ID,Content,Summary,Dataset
0,0,f49ee725a0360aa6881ed1f7999cc531885dd06a,New York police are concerned drones could bec...,Police have investigated criminals who have ri...,CNN/Daily Mail
1,1,808fe317a53fbd3130c9b7563341a7eea6d15e94,By . Ryan Lipman . Perhaps Australian porn sta...,Porn star Angela White secretly filmed sex act...,CNN/Daily Mail
2,2,98fd67bd343e58bc4e275bbb5a4ea454ec827c0d,"This was, Sergio Garcia conceded, much like be...",American draws inspiration from fellow country...,CNN/Daily Mail


In [6]:
# Show all column names
print("Columns:", df.columns.tolist())

# Usually the dataset has columns like:
# - 'article' / 'text' : the long document
# - 'summary' / 'headline' : the concise summary

# I'll assume typical names – you can manually adjust here
text_col = None
summary_col = None

for col in df.columns:
    if 'article' in col.lower() or 'text' in col.lower() or 'content' in col.lower():
        text_col = col
    if 'summary' in col.lower() or 'headline' in col.lower() or 'highlights' in col.lower():
        summary_col = col

if text_col is None or summary_col is None:
    print("Could not auto-detect. Please manually set text_col and summary_col.")
    # Manually set them after inspecting df.head()
else:
    print(f"Using text column: '{text_col}'")
    print(f"Using summary column: '{summary_col}'")

Columns: ['Unnamed: 0', 'ID', 'Content', 'Summary', 'Dataset']
Using text column: 'Content'
Using summary column: 'Summary'


In [7]:
# Keep only the two columns we need
df_clean = df[[text_col, summary_col]].copy()
df_clean.columns = ['article', 'summary']   # standardize names

# Remove any rows that have missing values
df_clean.dropna(inplace=True)

# For a quick demonstration, use only the first 2000 rows.
# You can increase this later (e.g., 5000) for better quality.
df_small = df_clean.head(2000).reset_index(drop=True)

print(f"Total usable rows: {len(df_clean)}")
print(f"Using {len(df_small)} rows for training (you can increase later).")
print("\nSample:")
print(df_small[['article', 'summary']].iloc[0])

Total usable rows: 870487
Using 2000 rows for training (you can increase later).

Sample:
article    New York police are concerned drones could bec...
summary    Police have investigated criminals who have ri...
Name: 0, dtype: object


In [8]:
# Model choice: "t5-small" is faster, "t5-base" gives better quality
MODEL_NAME = "t5-base"   # you can switch to "t5-small" for quicker runs

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded on {device}")

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on cuda


In [11]:
MAX_SOURCE_LENGTH = 512   # maximum tokens for article
MAX_TARGET_LENGTH = 128   # maximum tokens for summary

def preprocess_function(examples):
    # Add the prefix "summarize: " – T5 needs this to know the task
    inputs = ["summarize: " + doc for doc in examples["article"]]
    
    # Tokenize inputs (articles)
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding="max_length"
    )
    
    # Tokenize targets (summaries) – simpler version that works with T5
    labels = tokenizer(
        examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [12]:
dataset = Dataset.from_pandas(df_small)

# Split into train (90%) and test (10%)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset_raw = split_dataset["train"]
test_dataset_raw = split_dataset["test"]

print(f"Train size: {len(train_dataset_raw)}")
print(f"Test size: {len(test_dataset_raw)}")

# Apply tokenization (this may take a minute)
tokenized_train = train_dataset_raw.map(preprocess_function, batched=True, remove_columns=train_dataset_raw.column_names)
tokenized_test = test_dataset_raw.map(preprocess_function, batched=True, remove_columns=test_dataset_raw.column_names)

print("Tokenization complete!")

Train size: 1800
Test size: 200


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenization complete!


In [13]:
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Decode predictions (skip special tokens)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 (ignore index) with pad_token_id for decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Compute ROUGE
    result = rouge_metric.compute(
        predictions=decoded_preds, 
        references=decoded_labels, 
        use_stemmer=True
    )
    # Extract the fmeasure (harmonic mean) for each ROUGE variant
    return {k: round(v * 100, 2) for k, v in result.items()}

In [16]:
# Data collator – dynamically pads batches
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-summarization",
    eval_strategy="epoch",          # evaluate after each epoch
    save_strategy="epoch",               # save checkpoint each epoch
    learning_rate=3e-5,
    per_device_train_batch_size=4,       # adjust based on GPU memory (4 is safe for T5-base)
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    num_train_epochs=3,                  # start with 3 epochs
    predict_with_generate=True,          # use generate() for evaluation
    fp16=True,                           # mixed precision = faster + less memory
    logging_steps=50,
    save_total_limit=2,                  # keep only last 2 checkpoints
    report_to="none",                    # disable wandb / tensorboard
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start training!
print("🔄 Training started... (this will take ~10-20 minutes for 2000 samples on GPU)")
trainer.train()

print("✅ Fine-tuning finished!")

🔄 Training started... (this will take ~10-20 minutes for 2000 samples on GPU)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.167200,0.874604,16.420000,7.830000,13.600000,13.550000
2,1.910904,0.856250,22.390000,9.830000,18.120000,18.110000
3,1.966062,0.852175,23.530000,10.300000,19.040000,19.010000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning finished!


In [19]:
# Evaluate
eval_results = trainer.evaluate()
print("\n📈 Evaluation Results (ROUGE scores):")
for key, value in eval_results.items():
    if "rouge" in key:
        print(f"  {key}: {value:.2f}")


📈 Evaluation Results (ROUGE scores):
  eval_rouge1: 23.53
  eval_rouge2: 10.30
  eval_rougeL: 19.04
  eval_rougeLsum: 19.01


In [20]:
def generate_summary(article_text, max_length=150):
    input_text = "summarize: " + article_text
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=30,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Pick a few test samples
print("\n🔍 Sample generations:\n")
test_articles = test_dataset_raw.select(range(3))

for i, sample in enumerate(test_articles):
    article = sample["article"]
    reference = sample["summary"]
    generated = generate_summary(article)
    
    print(f"--- Example {i+1} ---")
    print(f"ARTICLE (first 300 chars): {article[:300]}...\n")
    print(f"REFERENCE SUMMARY: {reference}")
    print(f"GENERATED SUMMARY: {generated}")
    print("\n" + "-"*60 + "\n")


🔍 Sample generations:

--- Example 1 ---
ARTICLE (first 300 chars): (CNN) -- Two tractor-trailer trucks crashed and burst into flames Thursday on a bridge between the United States and Mexico, shutting a key border crossing and killing four people, police said. Police look at the aftermath of a fiery crash on a bridge linking Reynosa, Mexico, and Pharr, Texas. The c...

REFERENCE SUMMARY: NEW: Bridge reopens Friday morning after highway engineers give OK .
Four killed in chain reaction crash that started when two tractor-trailers collided .
Three killed in pickup truck that fell off bridge; 4th victim in car that hit one 18-wheeler .
Crash happened on bridge linking Pharr, Texas, and Reynosa, Mexico .
GENERATED SUMMARY: A pickup truck flipped off the Pharr-Reynosa International Bridge, killing three people . Two 18-wheelers and a minivan involved in the crash appeared to have Mexican plates . The bridge is normally open from 6 a.m. until midnight and is closed overnight .

-----------

In [31]:
# Save to a directory
model.save_pretrained("./my_t5_summarizer")
tokenizer.save_pretrained("./my_t5_summarizer")

# Optionally, create a zip file for download
!zip -r t5_summarizer_model.zip ./my_t5_summarizer
print("Model saved and zipped.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

updating: my_t5_summarizer/ (stored 0%)
updating: my_t5_summarizer/config.json (deflated 63%)
updating: my_t5_summarizer/generation_config.json (deflated 60%)
updating: my_t5_summarizer/tokenizer_config.json (deflated 83%)
updating: my_t5_summarizer/tokenizer.json (deflated 79%)
updating: my_t5_summarizer/model.safetensors (deflated 8%)
Model saved and zipped.
